In [1]:
import numpy as np
import numpy.typing as npt
from pathlib import Path
from astropy.time import Time
import sorts
from sorts import equidistant_sampling
from sorts.interpolation import Legendre8, Linear
from sorts.population import master_catalog, master_catalog_factor
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.tx_rx import Station
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.utils import to_datetime64_us, to_pydatetime
from sorts.controller_v2 import tracker_controller
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller import FenceScanController
from sorts import schedule_v2 as schedule
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import priority_scheduling
from sorts.simulation_v2 import StxMrxSimulation
from sorts.simulation_v2.observation import list_to_dataframe

# import for plottings
from IPython.display import display
import pandas as pd
import ipywidgets as widgets
import bokeh.plotting as bp
import bokeh.models as bokeh_models
import lets_plot as lp
import panel as pn
from sorts import plots

In [2]:
# disable pandas table wrapping
pd.set_option("display.expand_frame_repr", False)

# activate Bokeh output in Jupyter notebook
from bokeh.io import output_notebook, push_notebook
output_notebook()

# config and init lets-plot
lp.LetsPlot.setup_html()

# config and init panel
pn.extension("tabulator", comms='ipywidgets',
    # sizing_mode="stretch_width",
)

Loading BokehJS ...

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z

# the first set of value used, not much use now; kept for ref
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# a 1 sec long period, the sbobj should be very close to right up ahead of eiscat3d tx-0 station
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# an extended duration which expands around from the 1 sec period above
# the `control_slice_duration` is much longer than normal, practical radar `control_slice_duration`
# for easier debugging, inspection of scheduling/schedules
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

# same as above, but use more realistic 10ms `control_slice_duration`
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

tracked_spobj = SpaceObject(
    oid=-1,
    propagator=SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)

catalog_fpath = Path() / ".." /  ".." / "local_data" / "celn_20090501_00.sim"
_spobj_pop = master_catalog(
    catalog_fpath,
    propagator=SGP4,
    propagator_options={"settings": {"in_frame": "TEME", "out_frame": "ITRF"}},
)
rand_seed = 120389
# TODO: reduce the filter size to more sensible value
spobj_pop = master_catalog_factor(_spobj_pop, treshhold=5.0, seed=rand_seed)
spobjs = [tracked_spobj, *[spobj_pop.get_object(i) for i in range(spobj_pop.shape[0])]]

exp_detail_map: dict[int, ExperimentDetail] = {
    0: {
        "id":0,
        "coh_int_bandwidth":1.0,
        "ipp":1.0,
        "pulse_length":1.0,
        "power":5000000.0,
        "bandwidth":52.08333333333333,
        "duty_cycle":1.0,
        "noise_temp":150.0,
        "slice_duration":control_slice_duration
    },
    1: {
        "id":1,
        "coh_int_bandwidth":1.0,
        "ipp":1.0,
        "pulse_length":1.0,
        "power":5000000.0,
        "bandwidth":52.08333333333333,
        "duty_cycle":1.0,
        "noise_temp":150.0,
        "slice_duration":control_slice_duration
    }
}

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    to_datetime64_us(start_time),
    to_datetime64_us(end_time),
    control_slice_duration,
)
# time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - to_datetime64_us(epoch)
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = tracked_spobj.get_state(dsec_arr)

# tracker_ctrl = TrackerController.from_ecef_states(
#     tx_station=eiscat3d.tx[0],
#     # rx_stations=eiscat3d.rx[0:1],
#     rx_stations=eiscat3d.rx[0:2],
#     time=time_arr,
#     space_object_states=ecefs,
#     exp_detail=exp_detail_map[0],
# )

tracker_ctrl = TrackerController.from_space_object(
    spobj=tracked_spobj,
    epoch=epoch,
    tx_station=eiscat3d.tx[0],
    # rx_stations=eiscat3d.rx[0:1],
    rx_stations=eiscat3d.rx[0:2],
    exp_detail=exp_detail_map[0],
)

fence_scan_ctrl = FenceScanController.from_scan_spec(
    tx_station=eiscat3d.tx[0],
    # rx_stations=eiscat3d.rx[0:1],
    rx_stations=eiscat3d.rx[0:2],
    exp_detail=exp_detail_map[1],
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
    # scan_range=np.linspace(300e3, 1000e3, num=10, dtype=np.float64),
    scan_range=np.array([300e3], dtype=np.float64),
)

In [4]:
data_table = plots.space_object_population_table_plot(spobj_pop)

bp.show(data_table)

In [5]:
plot = plots.kepler_space_object_on_map(spobjs[0], epoch, start_time=start_time, end_time=end_time)
bp.show(plot)

In [6]:
# plot = plots.kepler_space_object_on_map(spobjs[4], epoch, start_time=start_time, end_time=end_time) # interesting s shape
# plot = plots.kepler_space_object_on_map(spobjs[5], epoch, start_time=start_time, end_time=end_time) # show be visible to eiscat
plot = plots.kepler_space_object_on_map(spobjs[17], epoch, start_time=start_time, end_time=end_time) # show be visible to eiscat
bp.show(plot)

/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 500 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 500 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 500 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [7]:
# plots.ecef_states_positions_plot(ecefs)

In [8]:
plot = tracker_ctrl.plot(start_time, end_time)
bp.show(plot)

In [9]:
tracker_schs = tracker_ctrl.generate(start_time, end_time)
tracker_tx_sch_df = schedule.to_dataframe(tracker_schs.tx_schedule)
tracker_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 04:03:00,-171.006800,32.375463,0,2025-01-01 04:04:00
1,2025-01-01 04:04:00,176.908697,45.133569,0,2025-01-01 04:05:00
2,2025-01-01 04:05:00,148.242673,58.357417,0,2025-01-01 04:06:00
3,2025-01-01 04:06:00,99.087086,59.010676,0,2025-01-01 04:07:00
4,2025-01-01 04:07:00,69.340087,46.622261,0,2025-01-01 04:08:00
5,2025-01-01 04:08:00,56.848433,34.318078,0,2025-01-01 04:09:00
6,2025-01-01 05:46:00,-108.596304,30.678467,0,2025-01-01 05:47:00
7,2025-01-01 05:47:00,-100.985872,43.559562,0,2025-01-01 05:48:00
8,2025-01-01 05:48:00,-81.551540,60.228457,0,2025-01-01 05:49:00
9,2025-01-01 05:49:00,-22.529323,69.383634,0,2025-01-01 05:50:00


In [10]:
# check schedule df memory usage (MB)
tracker_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.000612)

In [11]:
fence_schs = fence_scan_ctrl.generate(start_time, end_time)
fence_tx_sch_df = schedule.to_dataframe(fence_schs.tx_schedule)
fence_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [12]:
# check schedule df memory usage (MB)
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.008532)

In [13]:
plot = plots.azel_skyplot(tracker_schs.tx_schedule["pointing_az"], tracker_schs.tx_schedule["pointing_el"])
bp.show(plot)

In [14]:
plot = plots.azel_skyplot(tracker_schs.rx_schedules[1]["pointing_az"], tracker_schs.rx_schedules[1]["pointing_el"])
bp.show(plot)

In [15]:
plot = plots.azel_skyplot(fence_schs.tx_schedule["pointing_az"], fence_schs.tx_schedule["pointing_el"])
bp.show(plot)

In [16]:
schedule.to_dataframe(fence_schs.tx_schedule)

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [17]:
schedule.to_dataframe(fence_schs.rx_schedules[1])

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,61.879456,37.751736,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,60.514557,41.225552,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,58.919376,44.604865,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,57.053902,47.879705,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,54.868644,51.038432,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,52.302713,54.066972,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,49.282023,56.947828,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,45.718334,59.658862,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,41.510661,62.171873,1,2025-01-01 06:14:00


In [18]:
plot = plots.azel_skyplot(fence_schs.rx_schedules[1]["pointing_az"], fence_schs.rx_schedules[1]["pointing_el"])
bp.show(plot)

In [19]:
tx_master_sch = priority_scheduling([tracker_schs.tx_schedule, fence_schs.tx_schedule])
tx_master_sch

{'start_time': array(['2025-01-01T02:45:00.000000', '2025-01-01T02:46:00.000000',
        '2025-01-01T02:47:00.000000', '2025-01-01T02:48:00.000000',
        '2025-01-01T02:49:00.000000', '2025-01-01T02:50:00.000000',
        '2025-01-01T02:51:00.000000', '2025-01-01T02:52:00.000000',
        '2025-01-01T02:53:00.000000', '2025-01-01T02:54:00.000000',
        '2025-01-01T02:55:00.000000', '2025-01-01T02:56:00.000000',
        '2025-01-01T02:57:00.000000', '2025-01-01T02:58:00.000000',
        '2025-01-01T02:59:00.000000', '2025-01-01T03:00:00.000000',
        '2025-01-01T03:01:00.000000', '2025-01-01T03:02:00.000000',
        '2025-01-01T03:03:00.000000', '2025-01-01T03:04:00.000000',
        '2025-01-01T03:05:00.000000', '2025-01-01T03:06:00.000000',
        '2025-01-01T03:07:00.000000', '2025-01-01T03:08:00.000000',
        '2025-01-01T03:09:00.000000', '2025-01-01T03:10:00.000000',
        '2025-01-01T03:11:00.000000', '2025-01-01T03:12:00.000000',
        '2025-01-01T03:13:00.00000

In [20]:
tx_master_sch_df = schedule.to_dataframe(tx_master_sch)
tx_master_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
201,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
202,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
203,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
204,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [21]:
tx_master_sch_df[tx_master_sch_df["exp_num"] == 1]

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
201,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
202,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
203,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
204,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [22]:
sch = tx_master_sch

df = schedule.to_dataframe(sch)

start_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn["start_time"]].min().tz_localize("utc"),
    description='Start Time',
)
end_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn["start_time"]].min().tz_localize("utc") + np.timedelta64(5, "m"),
    description='End Time',
)

date_range_widget = widgets.HBox([start_datetime_widget, end_datetime_widget])
date_range_widget

In [23]:
# TODO: leverage `notebook_handle`, e.g. `plot_nbh = bp.show(plot, notebook_handle=True)` ?
plot = plots.schedule_plot(tx_master_sch, start_datetime_widget.value.replace(tzinfo=None), end_datetime_widget.value.replace(tzinfo=None))
bp.show(plot)

In [24]:
# TODO: these patching of station prop should be integrated into codebase
tx_station:Station = eiscat3d.tx[0]
tx_station.uid = ("eiscat3d", "stage1-array", "tx", "0")
rx_station_0:Station = eiscat3d.rx[0]
rx_station_0.uid = ("eiscat3d", "stage1-array", "rx", "0")
rx_station_1:Station = eiscat3d.rx[1]
rx_station_1.uid = ("eiscat3d", "stage1-array", "rx", "1")

rx_master_schs = [
    priority_scheduling(rx_schs)
    for rx_schs in zip(tracker_schs.rx_schedules, fence_schs.rx_schedules)
]


sim = StxMrxSimulation.from_spec(
    {
        "tx_station":tx_station,
        "tx_schedule":tx_master_sch,
        "rx_stations":[rx_station_0, rx_station_1],
        "rx_schedules":rx_master_schs,
        "exp_detail_map":exp_detail_map,
        "epoch":epoch,
        "start_time":start_time,
        "end_time":end_time,
        "space_objects":[o for i, o in enumerate(spobjs) if i in [0, 4, 5, 17]], # just picked a few from the whole list for now
        "dsec_sampler":lambda orbit, start_time, end_time: equidistant_sampling(
            orbit=orbit,
            start_t=(to_pydatetime(start_time) - to_pydatetime(epoch)).total_seconds(),
            end_t=(to_pydatetime(end_time) - to_pydatetime(epoch)).total_seconds(),
            max_dpos=1e3,
        ),
        "interpolator_class":Linear,
    }
)

In [25]:
plot = plots.radar_schedule_ecef_position_plot(ecefs=ecefs, ecefs_time=time_arr, sch = rx_master_schs[1])
bp.show(plot)

In [26]:
obss = sim.run()

 25%|██▌       | 1/4 [00:00<00:00,  3.71it/s]/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 61143 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 61143 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 61143 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
 50%|█████     | 2/4 [00:00<00:00,  4.49it/s]/home/tszhinh/projects/sorts/.venv/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 94960 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/

In [27]:
obss_df = list_to_dataframe(obss)
obss_df

,expps_experiment_detail,expps_space_object,expps_tx_station,expps_rx_station,expps_epoch,expps_time_range,id,snr,range,range_rx,range_rate,tx_k,rx_k
0,"{'id': 0, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object -1: <Time object: scale='utc' f...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f2a1690>,2004-01-01,"(2025-01-01 04:03:00, 2025-01-01 04:09:00)","0-(2025-01-01 04:03:00, 2025-01-01 04:09:00)","[274.05699408968036, 1074.8734156094126, 2722....","[2831576.052125955, 2314893.6754069906, 201113...","[1415788.0260629775, 1157446.8377034953, 10055...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.13201887541133717, 0.03804331926450933, 0...","[[-0.13201887541133717, 0.03804331926450933, 0..."
1,"{'id': 0, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object -1: <Time object: scale='utc' f...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f2a1690>,2004-01-01,"(2025-01-01 05:46:00, 2025-01-01 05:52:00)","0-(2025-01-01 05:46:00, 2025-01-01 05:52:00)","[205.51670544392863, 876.6708522728449, 2848.0...","[2970227.287985119, 2401913.877205393, 2007936...","[1485113.6439925595, 1200956.9386026964, 10039...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.815140306928066, -0.7113784531175842, -0....","[[-0.815140306928066, -0.7113784531175842, -0...."
2,"{'id': 0, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object -1: <Time object: scale='utc' f...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f353f90>,2004-01-01,"(2025-01-01 04:03:00, 2025-01-01 04:09:00)","1-(2025-01-01 04:03:00, 2025-01-01 04:09:00)","[28.524855280966733, 35.268193658836765, 4.311...","[2768094.580473012, 2250085.6349235764, 195156...","[1352306.5544100343, 1092638.7972200809, 94600...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.13201887541133717, 0.03804331926450933, 0...","[[-0.23862500652152016, -0.07316172438995032, ..."
3,"{'id': 0, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object -1: <Time object: scale='utc' f...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f353f90>,2004-01-01,"(2025-01-01 05:47:00, 2025-01-01 05:52:00)","1-(2025-01-01 05:47:00, 2025-01-01 05:52:00)","[47.7285135654155, 47.60136198013026, 21.14224...","[2457713.014717381, 2065891.010124272, 1951989...","[1256756.0761146846, 1061922.9118115907, 10018...","[1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.7113784531175842, -0.4911546406666414, -0...","[[-0.759924094787937, -0.5520882443490259, -0...."
4,"{'id': 1, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object 4: <Time object: scale='utc' fo...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f2a1690>,2004-01-01,"(2025-01-01 03:26:00, 2025-01-01 03:31:00)","0-(2025-01-01 03:26:00, 2025-01-01 03:31:00)","[225.30514956932367, 0.5768407104507702, 15416...","[2315375.4891885556, 1738834.6900628174, 14385...","[1157687.7445942778, 869417.3450314087, 719291...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.7889183838858262, -0.5818075971678227, -0...","[[-0.7889183838858262, -0.5818075971678227, -0..."
5,"{'id': 1, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object 4: <Time object: scale='utc' fo...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f2a1690>,2004-01-01,"(2025-01-01 05:07:00, 2025-01-01 05:12:00)","0-(2025-01-01 05:07:00, 2025-01-01 05:12:00)","[6721.483909076289, 9592.959575449184, 2731.69...","[2196480.3398549776, 1677239.6143807713, 14770...","[1098240.1699274888, 838619.8071903856, 738542...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.7652330361961409, -0.48120779184446455, 0...","[[-0.7652330361961409, -0.48120779184446455, 0..."
6,"{'id': 1, 'coh_int_bandwidth': 1.0, 'ipp': 1.0...",\nSpace object 4: <Time object: scale='utc' fo...,<sorts.radar.tx_rx.TX object at 0x788314ae72d0>,<sorts.radar.tx_rx.RX object at 0x78829f353f90>,2004-01-01,"(2025-01-01 03:26:00, 2025-01-01 03:31:00)","1-(2025-01-01 03:26:00

In [28]:
# plotting some observations on top of the map plot in 'radar_schedule_ecef_position_plot'

import typing as t
import bokeh.layouts as bokeh_layouts

space_object_id: int = -1
# rx_station = rx_station_1
ecefs= ecefs
sch = rx_master_schs[1]
ecefs_time=time_arr
observations = obss
# observation = obss[0]
# obs = observation

# define some field names
cn = t.cast(
    dict[t.Union[plots.RadarScheduleEcefPositionPlotColumnKey, t.Literal["obs_wmx", "obs_wmy"]], str],
    {k:k for k in t.get_args(plots.RadarScheduleEcefPositionPlotColumnKey)} | {
        "obs_wmx": "obs_wmx",
        "obs_wmy": "obs_wmy",
    }
)

df = plots._radar_schedule_ecef_position_plot_cds_df(ecefs=ecefs, ecefs_time=ecefs_time, sch=sch)

# merging some obss cols into the dataframe
df[cn["obs_wmx"]] = pd.NA
df[cn["obs_wmy"]] = pd.NA
# for obs in [o for o in obss if o.passage.space_object.oid == space_object_id and o.passage.rx_station == rx_station]:
# for obs in observations[0:1]:
for obs in observations[0:2]:
    mask = (df[cn["start_time"]] >= obs["experiment_passage"]["time_range"][0]) & (df[cn["end_time"]] <= obs["experiment_passage"]["time_range"][1])
    df = df.combine_first(
        pd.DataFrame({
            schedule.cn["start_time"]: df[cn["start_time"]][mask],
            cn["obs_wmx"]: df[cn["wmx"]][mask],
            cn["obs_wmy"]: df[cn["wmy"]][mask],
            # Observation.Cn["snr"]: obs.snr
        }),
    )

cds = bokeh_models.ColumnDataSource(df)

start_time = df[schedule.cn["start_time"]].min()
end_time = df[schedule.cn["end_time"]].max()

sch_plot, sch_plot_bar, *_ = plots._schedule_plot_from_cds(
    cds,
    start_time=start_time,
    end_time=end_time,
    y_range=df[schedule.cn["exp_num"]].dropna().unique(),
)
sch_plot_bar_select_tool = bokeh_models.BoxSelectTool()
sch_plot_bar.add_tools(sch_plot_bar_select_tool)
sch_plot_bar.toolbar.active_drag = sch_plot_bar_select_tool

skyplot = plots._azel_skyplot_from_cds(cds)
skyplot_select_tool = bokeh_models.LassoSelectTool()
skyplot.add_tools(skyplot_select_tool)
skyplot.toolbar.active_drag = skyplot_select_tool

ecefpos_plot = plots._ecef_states_positions_plot_from_cds(cds)
ecefpos_plot_select_tool = bokeh_models.LassoSelectTool()
ecefpos_plot.add_tools(ecefpos_plot_select_tool)


passage_glyph = bokeh_models.Scatter(x=cn["obs_wmx"], y=cn["obs_wmy"],  marker="square")
ecefpos_plot.add_glyph(cds, passage_glyph)

plot = bokeh_layouts.layout(
    [
        [sch_plot],
        [skyplot, ecefpos_plot],
    ]  # type: ignore
)

bp.show(plot)

In [29]:
# plotting spobj ecef pos on skyplot

from sorts.frames import cart_to_sph
from sorts.utils import wrap_azimuths_elevations

azelSkyplotColumnMapDefault = {
    "azimuth": "azimuth",
    "elevation": "elevation",
    "adj_azimuth": "adj_azimuth",
    "adj_elevation": "adj_elevation",
}

obss[0]["experiment_passage"]["time_range"]
# mask = (ecefs_time >= obss[0]["experiment_passage"]["time_range"][0]) & (ecefs_time <= obss[0]["experiment_passage"]["time_range"][1])

# sat_ecefs = ecefs[:, mask]
# sat_ecefs = ecefs[:, 73:90]
sat_ecefs = ecefs[:, 176:193]
tx_enu_spobj = tx_station.enu(sat_ecefs[0:3])
azelr = cart_to_sph(tx_enu_spobj, degrees=True)
azimuth,elevation = wrap_azimuths_elevations(azelr[0], azelr[1])

source = {
    "azimuth": azimuth,
    "elevation": elevation,
    "adj_azimuth": azimuth - 90,
    "adj_elevation": 90 - elevation,
}

cn = azelSkyplotColumnMapDefault

# make a plot and set the pixel aspect ratio to equal to the data aspect ratio
# (i.e. a circle in data will be a circle on screen)
plot = bp.figure(match_aspect=True)
# customize axis
plot.xaxis.fixed_location = 0
plot.yaxis.fixed_location = 0
plot.xaxis.ticker = [30, 60, 90]
plot.xaxis.major_label_overrides = {30: "60", 60: "30", 90: "0"}
plot.yaxis.ticker = []
plot.xaxis.axis_line_alpha = 0  # alternatively, `plot.xaxis.axis_line_color = "lightgray"`
plot.yaxis.axis_line_alpha = 0  # alternatively, `plot.yaxis.axis_line_color = "lightgray"`

# disable builtin grid, which is rectangular, we will draw a custom polar grid
plot.xgrid.visible = False
plot.ygrid.visible = False

# add a bit more padding to the plotting region, default is 0.1 (in ratio)
plot.x_range.range_padding = 0.15  # type: ignore
plot.y_range.range_padding = 0.15  # type: ignore

# disable per axis zoom when hovered on an axis
wheelZoomTool = next(
    (t for t in plot.toolbar.tools if isinstance(t, bokeh_models.WheelZoomTool))
)
wheelZoomTool.zoom_on_axis = False

# draw custom grid
ray_angles = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
plot.ray(
    x=[0 for _ in range(len(ray_angles))],
    y=[0 for _ in range(len(ray_angles))],
    length=90,
    angle=ray_angles,
    angle_units="deg",
    color="lightgray",
    line_width=1,
)
ring_sizes = [0, 30, 60, 90]  # in deg
plot.circle(
    y=[0 for _ in range(len(ring_sizes))],
    x=[0 for _ in range(len(ring_sizes))],
    radius=ring_sizes,
    color="lightgray",
    fill_alpha=0,
    line_width=1,
)

# add custom labels
for x, y, text, angle in [
    (0, 100, "N, 0°", 0),
    (100, 0, "E, 90°", -90),
    (0, -100, "S, 180°", 0),
    (-100, 0, "W, -90°", 90),
]:
    plot.add_layout(
        bokeh_models.Label(
            x=x,
            y=y,
            anchor="center",  # type: ignore
            text=text,
            angle=angle,
            angle_units="deg",
        )
    )

tf = bokeh_models.PolarTransform(
    angle=cn["adj_azimuth"],
    radius=cn["adj_elevation"],
    angle_units="deg",  # type: ignore
    direction="clock",
)

scatter = plot.scatter(
    x=tf.x,  # type: ignore
    y=tf.y,  # type: ignore
    source=source, # type: ignore
)

# add custom tooltips, only for the scatter plot/glyphs
plot.add_tools(
    bokeh_models.HoverTool(
        renderers=[scatter],
        tooltips=[
            ("index", "$index"),
            ("data (az, el)", f'(@{cn["azimuth"]}, @{cn["elevation"]})'),
        ],
    )
)

bp.show(plot)

In [30]:
# plotting observation radar pointings directly on map

from sorts.types import (
    EcefStates,
    Float64_as_m,
    Float64_as_deg,
    Datetime_Like,
    Datetime64_us,
    Timedelta64_us,
    Float64_as_sec,
)
from sorts.frames import ITRS_to_geodetic, sph_to_cart, enu_to_ecef, cart_to_sph
import pyproj

sat_ecefs = ecefs

station = obss[0]["experiment_passage"]["rx_station"]
cn = {
    "lat": "lat",
    "lon": "lon",
    "wmx": "wmx",
    "wmy": "wmy",
}
def azel_at_range_on_map(azimuths: npt.NDArray[Float64_as_deg], elevations: npt.NDArray[Float64_as_deg], ranges: npt.NDArray[Float64_as_m]):
    enus = sph_to_cart(np.array([azimuths, elevations, ranges]), degrees=True)

    ecefs = enu_to_ecef(station.ecef_lat, station.ecef_lon, station.ecef_alt, enus, degrees=True) + station.ecef[:, np.newaxis]

    geodetic_coords = ITRS_to_geodetic(ecefs[0], ecefs[1], ecefs[2])
    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326", "EPSG:3857"
    )  # World Geodetic System to Web Mercator
    lat = geodetic_coords[0]
    lon = geodetic_coords[1]
    wmx, wmy = transformer.transform(lat, lon)

    plot = bp.figure(
        x_axis_type="mercator",
        y_axis_type="mercator",
        match_aspect=True,
    )
    plot.add_tile("CartoDB Positron", retina=True)

    scatter = plot.scatter(x=cn["wmx"], y=cn["wmy"], source={
        cn["lat"]: lat,
        cn["lon"]: lon,
        cn["wmx"]: wmx,
        cn["wmy"]: wmy,
    })




    sat_geodetic_coords = ITRS_to_geodetic(sat_ecefs[0], sat_ecefs[1], sat_ecefs[2])
    sat_transformer = pyproj.Transformer.from_crs(
        "EPSG:4326", "EPSG:3857"
    )  # World Geodetic System to Web Mercator
    sat_lat = sat_geodetic_coords[0]
    sat_lon = sat_geodetic_coords[1]
    sat_wmx, sat_wmy = sat_transformer.transform(sat_lat, sat_lon)

    sat_scatter = plot.scatter(x="sat_wmx", y="sat_wmy", source={
        "sat_lat": sat_lat,
        "sat_lon": sat_lon,
        "sat_wmx": sat_wmx,
        "sat_wmy": sat_wmy,
    }, marker="square", color="black", alpha=0.2)




    # add custom tooltips, only for the scatter plot/glyphs
    plot.add_tools(
        bokeh_models.HoverTool(
            renderers=[scatter],
            tooltips=[
                ("index", "$index"),
                ("data (lat, lon)", f'(@{cn["lat"]}, @{cn["lon"]})'),
            ],
        )
    )
    return plot

rx_passage = schedule.filter_by_time_range(rx_master_schs[0], obss[0]["experiment_passage"]["time_range"])
rx_azler = cart_to_sph(np.array(obss[0]["rx_k"]), degrees=True)
plot = azel_at_range_on_map(rx_azler[0], rx_azler[1], obss[0]["range_rx"])
bp.show(plot)
